# Modelos supervisados
Regresión para estimar Y1 y Y2 y clasificación de calefacción. Ejecuta primero la preparación.

Se agrupa por las 12 geometrías (X1-X5) para mantener juntas sus variantes. Se reservan 3 geometrías para prueba y la elección de algoritmos usa solamente el entrenamiento.

In [1]:
from pathlib import Path
import sys
import pandas as pd
ROOT = Path.cwd()
if not (ROOT / "src").is_dir():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
print("Carpeta del proyecto:", ROOT.name)


Carpeta del proyecto: REC_ECBD_2026_Gonzalez_Ulises


## 1. Entrenar, comparar y ajustar
El código en src/modelos_supervisados.py utiliza Pipeline para ajustar el escalado y OneHotEncoder dentro de cada partición. GroupKFold con tres particiones compara algoritmos y parámetros. La clasificación usa un umbral fijo de 25 para Y1; es una regla didáctica que requiere revisión con el docente.

In [2]:
from src.modelos_supervisados import ajustar_modelos
resultados = ajustar_modelos()

Separación por geometría: 576 entrenamiento / 192 prueba.
Geometrías compartidas entre ambos conjuntos: 0.
Regresión seleccionada mediante validación: RandomForest {'modelo__max_depth': 8, 'modelo__min_samples_leaf': 1}
          modelo objetivo    MAE   RMSE      R2
    RandomForest       Y1 3.3304 3.6172  0.8064
    RandomForest       Y2 4.3288 4.7784  0.6958
Referencia_media       Y1 8.2677 8.8777 -0.1663
Referencia_media       Y2 8.5548 9.0930 -0.1016
Clasificación seleccionada mediante validación: Logistica {'modelo__C': 1.0}
            modelo  accuracy  precision_macro  recall_macro  F1_macro
         Logistica    0.7917           0.8214        0.8333    0.7913
Referencia_mayoria    0.3750           0.1875        0.5000    0.2727
Matriz de confusión [baja, alta]: [[72, 0], [40, 80]]


## 2. Revisar la separación
Las geometrías de prueba no aparecen en el entrenamiento. Los identificadores, grupos y cargas objetivo no se usan como entradas.

In [3]:
print("Geometrías de entrenamiento:", resultados["geometrias_entrenamiento"])
print("Geometrías de prueba:", resultados["geometrias_prueba"])
print("Geometrías compartidas:", resultados["geometrias_compartidas"])
print("Filas de entrenamiento:", resultados["entrenamiento_filas"])
print("Filas de prueba:", resultados["prueba_filas"])

Geometrías de entrenamiento: [2, 3, 4, 5, 6, 7, 8, 9, 12]
Geometrías de prueba: [1, 10, 11]
Geometrías compartidas: 0
Filas de entrenamiento: 576
Filas de prueba: 192


## 3. Comparar los errores con una referencia simple
MAE y RMSE se interpretan en la escala de la carga; no son porcentajes. R2 puede ser negativo. F1 macro da el mismo peso a ambas clases.

In [4]:
print(pd.read_csv(ROOT / "docs/metricas_regresion.csv").round(4).to_string(index=False))
print(pd.read_csv(ROOT / "docs/metricas_clasificacion.csv").round(4).to_string(index=False))
print("Matriz [baja, alta]:", resultados["clasificacion"]["matriz_confusion_baja_alta"])

          modelo objetivo    MAE   RMSE      R2
    RandomForest       Y1 3.3304 3.6172  0.8064
    RandomForest       Y2 4.3288 4.7784  0.6958
Referencia_media       Y1 8.2677 8.8777 -0.1663
Referencia_media       Y2 8.5548 9.0930 -0.1016
            modelo  accuracy  precision_macro  recall_macro  F1_macro
         Logistica    0.7917           0.8214        0.8333    0.7913
Referencia_mayoria    0.3750           0.1875        0.5000    0.2727
Matriz [baja, alta]: [[72, 0], [40, 80]]


![Regresión en prueba](../docs/figuras/04_regresion_prueba.png)

![Matriz de confusión](../docs/figuras/03_matriz_confusion.png)

## 4. Cargar un modelo y predecir
Este ejemplo utiliza un caso reservado para prueba. El valor real se muestra para compararlo con la estimación.

In [5]:
import joblib
from src.preparacion_datos import ENTRADAS
caso = pd.read_csv(ROOT / "data/processed/predicciones_prueba.csv").iloc[[0]]
modelo = joblib.load(ROOT / "modelos/modelo_regresion_cargas.joblib")
print("Entradas:")
print(caso[ENTRADAS].to_string(index=False))
print("Cargas reales:", caso[["Y1", "Y2"]].to_numpy()[0])
print("Cargas predichas:", modelo.predict(caso[ENTRADAS])[0].round(2))

Entradas:
 X1    X2    X3    X4  X5  X6  X7  X8
0.9 563.5 318.5 122.5 7.0   2 0.0   0
Cargas reales: [20.84 28.28]
Cargas predichas: [16.36 22.42]


## Interpretación pendiente
Esta es una evaluación inicial con una sola separación y datos simulados. Hay que revisar estabilidad en otras separaciones por geometría y los errores de clasificación. Los resultados no prueban ahorros en edificios reales.

Referencias: https://doi.org/10.24432/C51307 y https://scikit-learn.org/stable/modules/cross_validation.html.